In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
86,NaN,2025-26,1630573,Sam Hauser,Sam,1610612738,BOS,Boston Celtics,22501168,2026-04-09T00:00:00,BOS @ NYK,L,31.483333,2,7,0.286,2,6,0.333,0,0,0.000,0,2,2,3,0,0,0,0,0,0,6,1,12.9,0,0,13.0,1,31:29,1,118.6,119.0,119.0,114.9,115.3,115.3,3.8,3.7,3.7,0.143,0.00,30.0,0.000,0.071,0.032,0.0,0.0,0.429,0.429,0.101,0.104,89.46,89.19,74.33,89.19,0.047,58,2.0,7.0,38,84,0.452,16,43,0.372,14,16,0.875,13,29,42,23,11.0,4,1,2,16,17,106,-6.0,119.0,120.5,126.4,127.3,-7.4,-6.8,0.605,2.09,18.0,0.354,0.789,0.547,0.125,0.548,0.582,88.8,88.0,73.33,88,0.453,1610612752,NYK,New York Knicks,43,80,0.538,15,35,0.429,11,15,0.733,5,25,30,29,7.0,9,2,1,17,16,112,6.0,126.4,127.3,119.0,120.5,7.4,6.8,0.674,4.14,23.2,0.211,0.646,0.453,0.080,0.631,0.647,88.8,88.0,73.33,88,0.547,F,PF,28.0
87,NaN,2025-26,1629020,Jarred Vanderbilt,Jarred,1610612747,LAL,Los Angeles Lakers,22501170,2026-04-09T00:00:00,LAL @ GSW,W,25.596667,1,3,0.333,0,2,0.000,0,0,0.000,1,5,6,5,4,0,0,0,0,0,2,15,12.7,0,0,13.0,1,25:36,1,122.5,129.4,129.4,101.6,98.1,98.1,20.9,31.3,31.3,0.192,1.25,41.7,0.050,0.238,0.146,33.3,33.3,0.333,0.333,0.121,0.121,97.59,96.58,80.48,96.58,0.051,51,1.0,3.0,49,80,0.613,16,29,0.552,5,8,0.625,8,25,33,37,19.0,14,3,2,13,6,119,16.0,125.9,129.3,114.1,112.0,11.8,17.4,0.755,1.95,26.4,0.324,0.595,0.474,0.207,0.713,0.712,92.4,92.0,76.67,92,0.570,1610612744,GSW,Golden State Warriors,41,81,0.506,9,30,0.300,12,12,1.000,15,23,38,24,19.0,8,2,3,6,13,103,-16.0,114.1,112.0,125.9,129.3,-11.8,-17.4,0.585,1.26,18.0,0.405,0.676,0.526,0.207,0.562,0.597,92.4,92.0,76.67,92,0.430,NaN,PF,26.0
88,NaN,2025-26,1642880,Kam Jones,Kam,1610612754,IND,Indiana Pacers,22501167,2026-04-09T00:00:00,IND @ BKN,W,21.733333,2,7,0.286,0,2,0.000,0,0,0.000,1,2,3,6,4,0,0,1,2,1,4,9,12.6,0,0,13.0,1,21:44,1,116.8,119.6,119.6,99.7,97.9,97.9,17.2,21.7,21.7,0.286,1.50,35.3,0.045,0.083,0.065,23.5,23.5,0.286,0.286,0.208,0.203,102.96,102.70,85.58,102.70,0.016,46,2.0,7.0,51,98,0.520,8,31,0.258,13,18,0.722,13,53,66,43,12.0,4,5,5,16,21,123,29.0,117.2,118.3,87.6,90.4,29.7,27.9,0.843,3.58,26.9,0.260,0.828,0.579,0.115,0.561,0.581,106.1,104.0,86.67,104,0.714,1610612751,BKN,Brooklyn Nets,37,96,0.385,8,38,0.211,12,19,0.632,7,36,43,20,10.0,2,5,5,21,16,94,-29.0,87.6,90.4,117.2,118.3,-29.7,-27.9,0.541,2.00,14.8,0.172,0.740,0.421,0.096,0.427,0.450,106.1,104.0,86.67,104,0.286,NaN,SG,23.0
60,NaN,2025-26,1630551,Justin Champagnie,Justin,1610612764,WAS,Washing

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260410_233529.json


,home_team,away_team,commence_time,bookmakers
0,Miami Heat,Atlanta Hawks,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Boston Celtics,Orlando Magic,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,New York Knicks,Charlotte Hornets,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,San Antonio Spurs,Denver Nuggets,2026-04-13 00:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Los Angeles Clippers,Golden State Warriors,2026-04-13 00:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-10 10:24:09
US latest pull: 2026-04-10 10:22:52


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,James Harden,Over,22.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
1,Underdog,player_points,James Harden,Under,22.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
2,Underdog,player_points,Evan Mobley,Over,18.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
3,Underdog,player_points,Evan Mobley,Under,18.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
4,Underdog,player_points,Jonathan Kuminga,Over,12.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Micah Peavy: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,James Harden,AST,5.71,30.03,39.90,0.1537,0.1969,0.3426,0.88,5.91,13.67,"[0.2608922515001304, 0.1716443529007895, 0.217..."
1,Max Strus,AST,14.46,23.99,30.66,0.0581,0.1012,0.2744,0.84,2.43,8.41,"[0.2534854245880861, 0.0857632933104631, 0.093..."
2,Daniss Jenkins,AST,20.29,27.04,33.32,0.0866,0.1865,0.2943,1.76,5.04,9.81,"[0.2706359945872801, 0.1529051987767584, 0.328..."
3,Ausar Thompson,AST,17.84,26.61,35.30,0.0727,0.1251,0.2125,1.30,3.33,7.50,"[0.1921229586935638, 0.0, 0.124275062137531, 0..."
4,Miles Bridges,AST,14.23,29.46,37.44,0.0501,0.1105,0.2070,0.71,3.25,7.75,"[0.0899550224887556, 0.159846547314578, 0.0591..."
5,Jordan Poole,AST,19.00,30.30,37.21,0.1006,0.1616,0.2833,1.91,4.90,10.54,"[0.1140250855188141, 0.0602409638554216, 0.079..."
6,Tyrese Maxey,AST,7.97,30.11,41.28,0.1020,0.1792,0.2850,0.81,5.40,11.77,"[0.077359463641052, 0.1932100469224399, 0.2256..."
7,VJ Edgecombe,AST,27.53,36.50,40.36,0.0457,0.1314,0.2291,1.26,4.80,9.25,"[0.1448016217781639, 0.0, 0.2004008016032064, ..."
8,Adem Bona,AST,12.82,19.66,27.42,0.0639,0.0592,0.1251,0.82,1.16,3.43,"[0.1031991744066047, 0.0, 0.0633713561470215, ..."
9,Jalen Brunson,AST,6.09,28.74,38.61,0.1447,0.2292,0.3409,0.88,6.59,13.16,"[0.38860103626943, 0.4437869822485207, 0.16615..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
285,Kris Dunn,PTS,5.5,17.61,26.08,33.64,3.96,7.76,17.71,0.571,0.429
219,Kenrich Williams,PTS,14.5,19.25,29.06,37.97,4.13,10.39,25.74,0.387,0.613
55,Julian Champagnie,AST,1.5,15.28,25.86,33.72,0.47,1.46,5.02,0.534,0.466
156,Alperen Sengun,REB,9.5,11.31,28.93,36.72,2.16,9.63,17.59,0.309,0.691
141,Ryan Rollins,REB,4.5,22.37,33.38,36.66,0.74,4.70,9.46,0.444,0.556
47,Devin Carter,AST,5.5,18.00,24.69,32.89,0.87,3.08,7.91,0.252,0.748
88,James Harden,REB,4.5,5.71,30.03,39.90,0.24,4.01,10.48,0.355,0.645
236,Daeqwon Plowden,PTS,13.5,22.90,31.89,37.49,6.61,12.15,23.94,0.609,0.391
168,Ausar Thompson,PTS,9.0,17.84,26.61,35.30,6.07,10.63,21.76,0.525,0.475
5,Jordan Poole,AST,3.5,19.00,30.30,37.21,1.91,4.90,10.54,0.592,0.408


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
195,Josh Hart,PTS,11.5,10.84,27.76,36.20,2.67,8.97,20.88,0.322,0.678,PTS,Underdog,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-114.0,-111.0,0.533,0.526,14.3,14.0,9.45,2.8,2.5,-0.296,0.616,0.384,15.64,-27.01,0.4,0.6,0.53,0.63,31.13,5.02,0.17,0.07,16.17,6.0
177,Derrick White,PTS,16.5,12.20,28.54,37.48,3.80,13.86,35.62,0.229,0.771,PTS,Underdog,Orlando Magic,3.5,216.5,113.7,14.0,100.51,14.0,-137.0,-137.0,0.578,0.578,11.1,11.0,3.57,-5.4,-5.5,1.513,0.065,0.935,-88.76,61.75,0.2,0.1,0.20,0.49,34.94,1.74,0.15,0.06,16.40,5.0
201,Paolo Banchero,PTS,23.5,15.18,32.96,38.46,7.86,21.29,40.39,0.347,0.653,PTS,Underdog,Boston Celtics,-3.5,216.5,111.8,4.0,95.46,30.0,-105.0,-115.0,0.512,0.535,22.8,21.5,10.91,-0.7,-2.0,0.064,0.474,0.526,-7.46,-1.66,0.2,0.4,0.40,0.50,34.40,4.71,0.27,0.07,19.75,4.0
57,Derrick White,AST,5.5,12.20,28.54,37.48,0.79,4.16,9.81,0.246,0.754,AST,Underdog,Orlando Magic,3.5,216.5,113.7,14.0,100.51,14.0,-137.0,-137.0,0.578,0.578,3.8,4.0,1.81,-1.7,-1.5,0.939,0.174,0.826,-69.90,42.89,0.2,0.2,0.20,0.37,34.94,1.74,0.15,0.06,5.00,5.0
69,Mikal Bridges,REB,3.5,9.11,26.85,40.11,0.27,2.93,9.64,0.322,0.678,REB,Underdog,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-125.0,100.0,0.556,0.500,2.4,2.0,2.17,-1.1,-1.5,0.507,0.306,0.694,-44.92,38.80,0.2,0.2,0.40,0.45,32.43,4.22,0.16,0.06,3.00,6.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
191,Jalen Brunson,PTS,24.5,6.09,28.74,38.61,3.47,21.72,47.93,0.307,0.693,PTS,PrizePicks,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-106.0,-120.0,0.515,0.545,24.4,25.5,7.04,-0.1,1.0,0.014,0.494,0.506,-4.00,-7.23,0.6,0.6,0.60,0.58,36.30,4.11,0.29,0.05,28.00,5.0
62,Onyeka Okongwu,REB,7.5,14.47,28.36,36.89,2.30,8.38,17.00,0.451,0.549,REB,PrizePicks,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-118.0,-102.0,0.541,0.505,6.5,7.0,2.59,-1.0,-0.5,0.386,0.350,0.650,-35.34,28.73,0.4,0.4,0.47,0.56,29.43,6.27,0.18,0.04,7.40,5.0
106,Desmond Bane,REB,4.0,10.05,29.59,37.48,0.38,3.83,9.24,0.369,0.506,REB,PrizePicks,Boston Celtics,-3.5,216.5,111.8,4.0,95.46,30.0,-137.0,-137.0,0.578,0.578,4.4,5.0,2.37,0.4,1.0,-0.169,0.567,0.433,-1.91,-25.09,0.4,0.5,0.40,0.54,32.66,3.30,0.23,0.03,3.80,5.0
102,OG Anunoby,REB,5.5,9.16,29.02,39.84,0.38,4.27,10.54,0.363,0.637,REB,PrizePicks,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-137.0,-137.0,0.578,0.578,4.8,4.0,3.61,-0.2,-1.0,0.055,0.478,0.522,-17.31,-9.70,0.6,0.3,0.40,0.40,35.74,3.77,0.18,0.04,4.75,4.0
9,Jalen Brunson,AST,7.5,6.09,28.74,38.61,0.88,6.59,13.16,0.402,0.598,AST,PrizePicks,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-101.0,-127.0,0.502,0.559,8.7,8.5,2.98,1.2,1.0,-0.403,0.657,0.343,30.75,-38.69,0.8,0.7,0.73,0.42,36.30,4.11,0.29,0.05,6.40,5.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
87,Jonathan Kuminga,REB,5.5,16.56,21.88,29.71,0.95,3.73,10.35,0.363,0.637,REB,Betr DFS,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-137.0,-137.0,0.578,0.578,4.0,5.0,2.00,-1.0,0.0,0.500,0.309,0.691,-46.55,19.54,0.4,0.2,0.47,0.38,21.31,3.69,0.20,0.05,2.00,1.0
43,Mikal Bridges,AST,3.5,9.11,26.85,40.11,0.45,2.83,7.82,0.287,0.713,AST,Betr DFS,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,105.0,-123.0,0.488,0.552,3.1,2.0,2.47,-0.4,-1.5,0.162,0.436,0.564,-10.62,2.25,0.4,0.4,0.40,0.54,32.43,4.22,0.16,0.06,3.67,6.0
139,Jalen Brunson,REB,3.5,6.09,28.74,38.61,0.10,2.70,7.80,0.291,0.709,REB,Betr DFS,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,110.0,-139.0,0.476,0.582,2.5,1.5,2.12,-1.0,-2.0,0.472,0.318,0.682,-33.22,17.26,0.4,0.3,0.47,0.36,36.30,4.11,0.29,0.05,2.60,5.0
161,Jalen Johnson,PTS,17.5,10.58,31.40,39.53,4.67,17.75,38.99,0.552,0.448,PTS,Betr DFS,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-107.0,-112.0,0.517,0.528,20.2,19.0,5.61,-2.3,-3.5,0.410,0.341,0.659,-34.03,24.74,0.0,0.3,0.47,0.42,35.61,4.17,0.26,0.04,24.25,4.0
159,Jonathan Kuminga,PTS,12.5,16.56,21.88,29.71,7.13,13.68,26.05,0.510,0.490,PTS,Betr DFS,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-112.0,-111.0,0.528,0.526,10.8,11.0,6.76,-1.7,-1.5,0.251,0.401,0.599,-24.10,13.86,0.2,0.3,0.40,0.52,21.31,3.69,0.20,0.05,15.00,1.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
62,Onyeka Okongwu,REB,7.5,14.47,28.36,36.89,2.30,8.38,17.00,0.451,0.549,REB,DraftKings Pick6,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-118.0,-102.0,0.541,0.505,6.5,7.0,2.59,-1.0,-0.5,0.386,0.350,0.650,-35.34,28.73,0.4,0.4,0.47,0.56,29.43,6.27,0.18,0.04,7.40,5.0
150,Dyson Daniels,REB,7.5,14.95,29.88,38.18,1.08,5.29,12.24,0.339,0.661,REB,DraftKings Pick6,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,113.0,-144.0,0.469,0.590,7.5,6.5,3.78,0.0,-1.0,0.000,0.500,0.500,6.50,-15.28,0.4,0.3,0.33,0.34,32.59,5.81,0.16,0.03,6.33,6.0
69,Mikal Bridges,REB,3.5,9.11,26.85,40.11,0.27,2.93,9.64,0.322,0.678,REB,DraftKings Pick6,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-125.0,100.0,0.556,0.500,2.4,2.0,2.17,-1.1,-1.5,0.507,0.306,0.694,-44.92,38.80,0.2,0.2,0.40,0.45,32.43,4.22,0.16,0.06,3.00,6.0
159,Jonathan Kuminga,PTS,12.5,16.56,21.88,29.71,7.13,13.68,26.05,0.510,0.490,PTS,DraftKings Pick6,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-112.0,-111.0,0.528,0.526,10.8,11.0,6.76,-1.7,-1.5,0.251,0.401,0.599,-24.10,13.86,0.2,0.3,0.40,0.52,21.31,3.69,0.20,0.05,15.00,1.0
261,Desmond Bane,PTS,20.5,10.05,29.59,37.48,4.87,19.62,35.74,0.409,0.591,PTS,DraftKings Pick6,Boston Celtics,-3.5,216.5,111.8,4.0,95.46,30.0,-103.0,-105.0,0.507,0.512,20.6,19.5,4.67,0.1,-1.0,-0.021,0.508,0.492,0.12,-3.94,0.6,0.5,0.47,0.48,32.66,3.30,0.23,0.03,15.00,5.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
216,Christian Braun,PTS,12.5,15.88,28.32,38.09,4.79,10.99,22.90,0.468,0.532,PTS,PrizePicks,San Antonio Spurs,-2.0,238.5,110.2,3.0,100.71,12.0,-114.0,-114.0,0.533,0.533,13.0,12.5,4.27,0.5,0.0,-0.117,0.547,0.453,2.68,-14.96,0.6,0.5,0.53,0.56,31.08,5.78,0.15,0.04,12.75,4.0
194,OG Anunoby,PTS,16.5,9.16,29.02,39.84,3.27,15.43,36.06,0.471,0.529,PTS,Underdog,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-137.0,-137.0,0.578,0.578,17.2,16.5,7.54,1.7,1.0,-0.225,0.589,0.411,1.89,-28.90,0.6,0.6,0.67,0.56,35.74,3.77,0.18,0.04,18.75,4.0
159,Jonathan Kuminga,PTS,12.5,16.56,21.88,29.71,7.13,13.68,26.05,0.510,0.490,PTS,Underdog,Miami Heat,-6.5,243.5,113.7,13.0,104.22,1.0,-112.0,-111.0,0.528,0.526,10.8,11.0,6.76,-1.7,-1.5,0.251,0.401,0.599,-24.10,13.86,0.2,0.3,0.40,0.52,21.31,3.69,0.20,0.05,15.00,1.0
109,Julian Champagnie,REB,5.0,15.28,25.86,33.72,1.25,4.97,12.62,0.492,0.401,REB,Betr DFS,Denver Nuggets,2.0,238.5,116.0,21.0,99.46,20.0,-137.0,-137.0,0.578,0.578,6.1,6.0,2.13,1.1,1.0,-0.516,0.697,0.303,20.58,-47.58,0.4,0.5,0.47,0.37,26.50,3.01,0.15,0.06,5.50,6.0
138,Josh Hart,REB,7.5,10.84,27.76,36.20,1.37,6.90,14.58,0.412,0.588,REB,Betr DFS,Charlotte Hornets,7.5,215.5,113.6,12.0,97.65,26.0,-137.0,-137.0,0.578,0.578,6.2,6.0,2.53,-1.3,-1.5,0.514,0.304,0.696,-47.41,20.40,0.2,0.2,0.33,0.56,31.13,5.02,0.17,0.07,8.17,6.0


### Get top EVs for 2 legs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 71  |  Pairs: 21  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 34  |  Pairs: 51  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 51  |  Pairs: 15  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 48  |  Pairs: 8  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [18]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 71  |  Triples: 159  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 34  |  Triples: 341  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 48  |  Triples: 43  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 51  |  Triples: 146  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
